# T-Tests And Mann-Whitney U

This notebook explains the two-group tests used in the project:

- Welch t-test
- Student two-sample t-test
- Mann-Whitney U

Use these when comparing two groups, such as event weeks vs non-event weeks.

## Formula

Welch's test statistic is:

$$
t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{s_1^2/n_1 + s_2^2/n_2}}
$$

Mann-Whitney U is rank-based, so it asks whether one group tends to have larger values than the other.

## Coding Agent Prompt

```text
Given two groups of geography-week outcomes, decide whether Welch t-test, Student t-test, or Mann-Whitney U is appropriate. State the null hypothesis, assumptions, sample sizes, effect size, p-value interpretation, and why the result is association only.
```

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "processed").exists())
DATA = ROOT / "data" / "processed" / "us"

## Project Context

In this project, `has_event` separates geography-weeks into:

- event weeks
- non-event weeks

The null hypothesis is that the two groups have equal economic activity.

In [ ]:
panel = pd.read_csv(DATA / "modeling" / "county_event_panel.csv")
event = panel.loc[panel["has_event"], "revenue_all"].dropna()
non_event = panel.loc[~panel["has_event"], "revenue_all"].dropna()

print(len(event), len(non_event))
print(event.mean(), non_event.mean(), event.mean() - non_event.mean())

In [ ]:
welch = stats.ttest_ind(event, non_event, equal_var=False)
student = stats.ttest_ind(event, non_event, equal_var=True)
mann_whitney = stats.mannwhitneyu(event, non_event, alternative="two-sided")

pd.DataFrame([
    {"test": "Welch t-test", "statistic": welch.statistic, "p_value": welch.pvalue},
    {"test": "Student t-test", "statistic": student.statistic, "p_value": student.pvalue},
    {"test": "Mann-Whitney U", "statistic": mann_whitney.statistic, "p_value": mann_whitney.pvalue},
])

## Assumptions

Welch t-test:

- independent observations
- approximately continuous outcome
- does not require equal variance
- more robust than Student t-test when group variances differ

Student two-sample t-test:

- independent observations
- approximately continuous outcome
- assumes equal variance across groups

Mann-Whitney U:

- independent observations
- ordinal or continuous outcome
- compares distributions/ranks rather than strictly comparing means

## Statistical Power Note

Power depends on sample size, effect size, variance, and alpha threshold. Large samples can make very small differences statistically significant. Always read the effect size and the p-value together.

## Interpretation

A small p-value means the observed group difference is unlikely under the equal-groups null. It does not prove sports events caused the difference.